In [0]:
bookings_df = spark.table("dev.spark_db.bookings")
facilities_df = spark.table("dev.spark_db.facilities")
members_df = spark.table("dev.spark_db.members")

df = bookings_df.join(facilities_df , bookings_df.facility_id == facilities_df.facility_id , "inner")\
                      .join(members_df , bookings_df.member_id == members_df.member_id , 'left')\
                      .select(bookings_df.member_id.alias("member_id") , "booking_id" , 
                               "first_name" , "last_name" , "guest_cost" , "member_cost" , "start_time" , "facility_name" , "slots")\
                      .selectExpr("booking_id",
                                 "case when member_id == 0 then 'Guest Name' else concat_ws(' ' , first_name , last_name) end as member_name",
                                 "facility_name",
                                 "start_time",
                                 "case when member_id == 0 then slots * guest_cost else slots * member_cost end as member_amount")
    
df.display()

In [0]:
from pyspark.sql.functions import sum , avg

df.selectExpr("sum(member_amount) as sum" , "avg(member_amount) as avg").display()
df.select(sum(df.member_amount).alias("sum") , avg(df.member_amount).alias("avg")).display()

In [0]:
df.agg(sum("member_amount").alias("sum"),
       avg("member_amount").alias("avg"))\
        .display()